<a href="https://colab.research.google.com/github/Omkar210/Statistics-and-ML/blob/main/Day22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
from numpy import random
from collections import Counter
from numpy.linalg import inv
from numpy.linalg import eig
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import pylab
from pylab import legend
from pylab import plot, show, title, xlabel, ylabel
import scipy
from scipy import stats
from scipy.stats import binom,poisson,norm,t,expon,f
from sklearn.model_selection import train_test_split
import statsmodels
from statsmodels import stats
from statsmodels.stats import weightstats as ssw
import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.stats.multicomp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.stats
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chisquare
from scipy.stats import chi2_contingency
from statsmodels.stats import rates
from statsmodels.stats.rates import test_poisson
from statsmodels.stats.rates import test_poisson_2indep
from scipy.stats import chi2
from scipy.stats import f
import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.stats.multicomp
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
# from category_encoders import BinaryEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import log_loss
from statsmodels.discrete.discrete_model import MNLogit
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
import statsmodels.formula.api as smf
from statsmodels.discrete.discrete_model import Poisson as psn
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
import os
print(os.getcwd())
os.chdir('/content/drive/MyDrive/CDAC')
os.getcwd()

/content


'/content/drive/MyDrive/CDAC'

# XGBoost Notes

## Loss Function (Regression)

For regression, XGBoost commonly uses Mean Squared Error (MSE):

$$
L = \frac{1}{2N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2
$$

### Derivatives

**First Derivative (Gradient)**

The gradient represents the prediction error (residual).

$$
g_i = \hat{y}_i - y_i
$$

**Second Derivative (Hessian)**

For MSE loss:

$$
h_i = 1
$$

The Hessian is constant for all observations.

---

## Step 1: Calculate Base Prediction

The initial prediction (base score) is the average of the target values.

$$
\text{Base Prediction}
=
\frac{\sum y_i}{N}
$$

---

## Step 2: Compute Gradient and Hessian

For every training sample:

### Gradient

$$
g_i = \hat{y}_i - y_i
$$

### Hessian

$$
h_i = 1
$$

---

## Step 3: Similarity Score / Leaf Weight

XGBoost calculates the optimal leaf value using:

$$
w_j
=
-\frac{\sum g_i}
{\sum h_i+\lambda}
$$

Where:

- $\sum g_i$ = Sum of gradients in the node
- $\sum h_i$ = Sum of Hessians in the node
- $\lambda$ = L2 regularization parameter

### Interpretation

The numerator measures total prediction error in the node.

The denominator penalizes large leaf values using regularization.

---

## Step 4: Calculate Left and Right Node Statistics

For each candidate split:

### Left Node

$$
G_L = \sum g_i
$$

$$
H_L = \sum h_i
$$

### Right Node

$$
G_R = \sum g_i
$$

$$
H_R = \sum h_i
$$

Where:

- $G_L$ = Sum of gradients in left node
- $H_L$ = Sum of Hessians in left node
- $G_R$ = Sum of gradients in right node
- $H_R$ = Sum of Hessians in right node

---

## Step 5: Compute Leaf Values

### Left Leaf Value

$$
w_L
=
-\frac{G_L}{H_L+\lambda}
$$

### Right Leaf Value

$$
w_R
=
-\frac{G_R}{H_R+\lambda}
$$

These values become the predictions of the leaf nodes.

---

## Step 6: Calculate Gain

Gain measures how much a split improves the objective function.

$$
Gain =
\frac{1}{2}
\left(
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
-
\frac{(G_L+G_R)^2}{H_L+H_R+\lambda}
\right)
-\gamma
$$

### Interpretation

- Higher gain ⇒ Better split
- Lower gain ⇒ Poor split
- Negative gain ⇒ Split is rejected

---

## Simplified Gain Formula

Assuming:

$$
\gamma = 0
$$

and

$$
\lambda = 1
$$

then:

$$
Gain =
\frac{1}{2}
\left(
\frac{G_L^2}{H_L+1}
+
\frac{G_R^2}{H_R+1}
-
\frac{(G_L+G_R)^2}{H_L+H_R+1}
\right)
$$

---

# Gamma ($\gamma$)

Gamma controls the minimum gain required to perform a split.

### Effects of Gamma

- If Gain < $\gamma$, the split is rejected.
- Larger Gamma makes splitting harder.
- Fewer splits lead to simpler trees.
- Helps prevent overfitting.

### Extreme Cases

**Very High Gamma**

- Very few splits
- Simpler model
- Risk of underfitting

**Very Low Gamma**

- Many splits
- Complex trees
- Risk of overfitting

---

# Lambda ($\lambda$)

Lambda is the L2 regularization parameter.

It appears in:

$$
w_j
=
-\frac{\sum g_i}
{\sum h_i+\lambda}
$$

### Effects of Lambda

- Increases denominator
- Reduces leaf weights
- Makes predictions less extreme
- Reduces variance
- Helps prevent overfitting

### Extreme Cases

**High Lambda**

- Strong regularization
- Smaller leaf values
- Conservative model
- May lead to underfitting

**Low Lambda**

- Weak regularization
- Larger leaf values
- More flexible model
- May lead to overfitting

---

# Complete XGBoost Training Flow

1. Calculate base prediction using target mean.
2. Compute gradients and Hessians.
3. Try every possible split.
4. Calculate left and right node gradients and Hessians.
5. Compute Gain for each split.
6. Select the split with highest positive Gain.
7. Calculate leaf weights.
8. Grow the tree recursively.
9. Add the tree's prediction to previous predictions.
10. Repeat for the specified number of boosting rounds.

---

# Alpha ($\alpha$)

Alpha is the **L1 Regularization** parameter in XGBoost.

Unlike Lambda (L2 regularization), Alpha penalizes the absolute value of leaf weights.

Its main purpose is to reduce the impact of small gradients and encourage simpler models.

---

## Leaf Weight with L1 Regularization

Without Alpha:

$$
w_j
=
-\frac{\sum g_i}
{\sum h_i+\lambda}
$$

With Alpha:

$$
w_j
=
-\frac{\operatorname{sgn}(G)\max(|G|-\alpha,0)}
{H+\lambda}
$$

Where:

$$
G=\sum g_i
$$

$$
H=\sum h_i
$$

---

## Soft Thresholding

Alpha applies a soft-threshold operation to the gradient sum.

$$
\operatorname{SoftThreshold}(G,\alpha)
=
\operatorname{sgn}(G)\max(|G|-\alpha,0)
$$

---

## Effect of Alpha

- Reduces the effective gradient magnitude.
- Produces smaller leaf weights.
- Reduces the correction made to previous predictions.
- Makes the model more conservative.
- Helps prevent overfitting.

---

## Extreme Cases

### Small Alpha

$$
\alpha \approx 0
$$

- Almost no L1 regularization.
- Larger leaf weights.
- More aggressive updates.

### Large Alpha

$$
\alpha \gg 0
$$

- Strong L1 regularization.
- Many leaf weights become zero.
- Simpler model.
- May lead to underfitting if too large.

---

## Comparison of Regularization Parameters

### Lambda (L2)

$$
w_j
=
-\frac{G}
{H+\lambda}
$$

- Increases denominator.
- Shrinks leaf weights smoothly.
- Rarely makes weights exactly zero.

### Alpha (L1)

$$
w_j
=
-\frac{\operatorname{sgn}(G)\max(|G|-\alpha,0)}
{H+\lambda}
$$

- Reduces numerator.
- Can make leaf weights exactly zero.
- Performs feature/leaf pruning effect.

---

## Interview Explanation

Alpha applies L1 regularization in XGBoost. It reduces the effective gradient magnitude before calculating leaf weights. Smaller gradients may become zero after thresholding, leading to smaller or zero leaf weights. As a result, each tree makes smaller corrections to previous predictions, helping reduce overfitting.

# XGBoost Interview Explanation

## What is XGBoost?

XGBoost (Extreme Gradient Boosting) is an advanced implementation of Gradient Boosting that builds decision trees sequentially. Each new tree tries to correct the errors made by the previous trees.

It is widely used because it is:

- Fast
- Scalable
- Highly accurate
- Supports regularization
- Handles missing values automatically

---

## Core Idea

Instead of building one large tree, XGBoost builds many small trees.

Each new tree focuses on the mistakes (residuals) made by the previous trees.

The final prediction is:

$$
\hat{y}
=
\sum_{k=1}^{K} f_k(x)
$$

where:

- $f_k(x)$ = prediction from tree $k$
- $K$ = total number of trees

---

## Example

Suppose we want to predict house prices.

Actual prices:

| House | Actual Price |
|---------|------------|
| A | 100 |
| B | 120 |
| C | 140 |

### Step 1: Base Prediction

Take the average:

$$
\frac{100+120+140}{3}
=
120
$$

Initial prediction for all houses:

| House | Actual | Prediction |
|---------|---------|------------|
| A | 100 | 120 |
| B | 120 | 120 |
| C | 140 | 120 |

---

## Step 2: Calculate Residuals

Residual:

$$
Residual
=
Actual - Prediction
$$

| House | Residual |
|---------|---------|
| A | -20 |
| B | 0 |
| C | 20 |

The next tree tries to predict these residuals.

---

## Step 3: Build First Tree

The tree learns patterns in the residuals.

Its output may be:

| House | Tree Output |
|---------|------------|
| A | -15 |
| B | 0 |
| C | 15 |

Update predictions:

$$
New\ Prediction
=
Old\ Prediction + Tree\ Output
$$

| House | Updated Prediction |
|---------|------------|
| A | 105 |
| B | 120 |
| C | 135 |

Predictions become closer to actual values.

---

## Step 4: Repeat

Again calculate residuals.

Again build a new tree.

Again update predictions.

Continue until:

- Desired number of trees is reached
- Improvement becomes very small

---

# How XGBoost Improves Gradient Boosting

XGBoost adds:

### 1. Regularization

Controls model complexity.

#### Lambda (L2)

$$
w_j
=
-\frac{G}{H+\lambda}
$$

Reduces large leaf weights.

#### Alpha (L1)

$$
w_j
=
-\frac{\operatorname{sgn}(G)\max(|G|-\alpha,0)}
{H+\lambda}
$$

Can force leaf weights to become zero.

---

### 2. Gain-Based Splitting

Every possible split is evaluated.

$$
Gain =
\frac{1}{2}
\left(
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
-
\frac{(G_L+G_R)^2}{H_L+H_R+\lambda}
\right)
-\gamma
$$

The split with highest gain is selected.

---

### 3. Gamma Pruning

$$
Gain > \gamma
$$

Only perform a split if gain exceeds gamma.

This prevents unnecessary tree growth.

---

### 4. Second-Order Optimization

Traditional Gradient Boosting uses:

- First derivative (Gradient)

XGBoost uses:

- First derivative (Gradient)
- Second derivative (Hessian)

This allows faster and more accurate optimization.

---

# Important Terms

## Gradient

Measures prediction error.

For regression:

$$
g_i
=
\hat{y}_i-y_i
$$

---

## Hessian

Measures curvature of the loss function.

For MSE:

$$
h_i
=
1
$$

---

## Similarity Score

Measures node quality.

$$
Similarity\ Score
=
\frac{G^2}{H+\lambda}
$$

Higher similarity score indicates a better node.

---

## Leaf Weight

Prediction assigned to a leaf.

$$
w_j
=
-\frac{G}{H+\lambda}
$$

---

# Why XGBoost Performs Better

1. Uses Gradient + Hessian information.
2. Uses regularization to reduce overfitting.
3. Uses gain to choose the best split.
4. Automatically prunes weak branches.
5. Handles missing values efficiently.
6. Supports parallel processing.
7. Works well on structured/tabular datasets.

---

# Common Interview Questions

### Why is XGBoost called Boosting?

Because every new tree improves or boosts the performance of the previous trees by correcting their errors.

---

### Why does XGBoost use Hessian?

The Hessian provides curvature information about the loss function, helping XGBoost find better leaf values and split decisions.

---

### What is Gain?

Gain measures how much a split improves the objective function.

Higher gain means a better split.

---

### What is Gamma?

Gamma is the minimum gain required to perform a split.

Higher gamma leads to fewer splits and less overfitting.

---

### What is Lambda?

Lambda is L2 regularization.

It reduces large leaf weights and prevents overfitting.

---

### What is Alpha?

Alpha is L1 regularization.

It can shrink leaf weights to zero and simplify the model.

---

# 30-Second Interview Answer

"XGBoost is an optimized Gradient Boosting algorithm that builds decision trees sequentially. Each new tree learns from the errors made by previous trees. Unlike traditional Gradient Boosting, XGBoost uses both gradients and Hessians for optimization and includes regularization through Alpha and Lambda to prevent overfitting. It selects splits using a Gain formula and prunes weak splits using Gamma, making it one of the most accurate and efficient algorithms for structured data."

# XGBoost Interview Explanation

If an interviewer asks **"Explain XGBoost"**, a strong answer is:

> XGBoost (Extreme Gradient Boosting) is an optimized implementation of Gradient Boosting that builds decision trees sequentially. Each new tree is trained to correct the errors made by the previous trees. It uses first-order gradients and second-order derivatives (Hessians) to determine the best splits and leaf values, making it faster and more accurate than traditional Gradient Boosting.

---

# How XGBoost Works (Step by Step)

## Step 1: Initial Prediction

For regression, XGBoost starts with a base prediction.

Usually:

$$
\hat{y}_0 = \frac{\sum y_i}{N}
$$

Example:

| Actual Values |
| ------------- |
| 10            |
| 15            |
| 20            |
| 25            |

Base prediction:

$$
\frac{10+15+20+25}{4}=17.5
$$

Every row initially gets prediction = 17.5.

---

## Step 2: Calculate Gradients and Hessians

For Loss :

$$
L=\frac{1}{2N}\sum(y_i-\hat{y}_i)^2
$$

Gradient:

$$
g_i = \hat{y}_i-y_i
$$

Hessian:

$$
h_i = 1
$$

Gradient tells us:

* How wrong the prediction is
* Direction of correction needed

Hessian tells us:

* Curvature of loss function
* Confidence in gradient

---

## Step 3: Find Best Split

For every possible split:

Calculate:

* Left node gradient sum
* Right node gradient sum
* Left node hessian sum
* Right node hessian sum

Then compute Gain.

---

## Step 4: Gain Calculation

XGBoost chooses the split with maximum gain.

$$
Gain =
\frac{1}{2}
\left(
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
-
\frac{(G_L+G_R)^2}{H_L+H_R+\lambda}
\right)
-\gamma
$$

Where:

* $G_L$ = Left node gradient sum
* $H_L$ = Left node hessian sum
* $G_R$ = Right node gradient sum
* $H_R$ = Right node hessian sum
* $\lambda$ = L2 regularization
* $\gamma$ = Minimum gain required for split

### Interview Line

> Gain tells us how much the split improves the objective function. XGBoost selects the split with the highest positive gain.

---

## Step 5: Calculate Leaf Weight

After selecting the best split:

Leaf value:

$$
w_j
= -\frac{\sum g_i}
{\sum h_i+\lambda}
$$

This becomes the prediction of that leaf.

### Interview Line

> The leaf weight determines how much the prediction should be adjusted for all samples reaching that leaf.

---

## Step 6: Update Predictions

New prediction:

$$
\hat{y}_{new} =
\hat{y}_{old}
+
\eta w_j
$$

Where:

* $\eta$ = Learning Rate

### Interview Line

> Learning rate controls how much of the new tree's prediction is added to the existing prediction.

---

## Step 7: Repeat

XGBoost builds multiple trees sequentially.

Each tree focuses on correcting remaining errors.

Final prediction:

$$
\hat{y}
=
\sum_{k=1}^{M}
\eta f_k(x)
$$

Where:

* (M) = Number of trees
* $f_k(x)$ = Prediction from tree (k)

---

# Why XGBoost is Better than Gradient Boosting

| Gradient Boosting     | XGBoost                           |
| --------------------- | --------------------------------- |
| Uses first derivative | Uses first and second derivatives |
| Less regularization   | Strong regularization             |
| Slower                | Faster                            |
| No built-in pruning   | Built-in pruning                  |
| More overfitting      | Better control of overfitting     |
| Sequential processing | Highly optimized implementation   |

---

# Important Hyperparameters

## Learning Rate (η)

Controls update size.

Small η:

* Slower learning
* Better generalization

Large η:

* Faster learning
* Risk of overfitting

---

## Gamma (γ)

Minimum gain required for a split.

High Gamma:

* Fewer splits
* Simpler tree
* Prevents overfitting

Low Gamma:

* More splits
* Complex tree

---

## Lambda (λ)

L2 Regularization.

$$
w_j
= -\frac{G}{H+\lambda}
$$

High Lambda:

* Smaller leaf weights
* More conservative model

---

## Alpha (α)

L1 Regularization.

$$
w_j
=-\frac{\sum (G)\max(|G|-\alpha,0)}
{H+\lambda}
$$

High Alpha:

* More leaf weights become zero
* Simpler model

---

# Common Interview Questions

### Why does XGBoost use Hessian?

> Hessian provides second-order information about the loss function, allowing more accurate optimization and better split decisions than methods using only gradients.

### Why is XGBoost faster?

> Due to parallelized split finding, optimized memory usage, tree pruning, cache awareness, and efficient handling of sparse data.

### What is Gain?

> Gain measures improvement in the objective function after a split. The split with highest positive gain is selected.

### What is the difference between Gradient Boosting and XGBoost?

> XGBoost adds second-order optimization, regularization, pruning, sparse-data handling, and parallelization, making it more accurate and efficient.

### What happens if Gamma is very high?

> Most splits will be rejected, producing simpler trees and potentially causing underfitting.

### What happens if Lambda is very high?

> Leaf weights become very small, making the model conservative and potentially causing underfitting.

---

### 30-Second Interview Answer

> XGBoost is an advanced gradient boosting algorithm that builds decision trees sequentially. It starts with an initial prediction, calculates gradients and Hessians from the loss function, evaluates possible splits using a gain formula, and assigns optimal leaf weights using regularization. Each new tree corrects the errors of previous trees. XGBoost is widely used because it provides high accuracy, handles overfitting through regularization, supports parallel processing, and efficiently handles missing and sparse data.


In [ ]:
df = pd.read_excel('CDAC_DataBook.xlsx', sheet_name='diabetes')
df = df[['Glucose','BloodPressure','Age','DietType','Outcome']]
diet_dummy = pd.get_dummies(df.DietType, drop_first=True, prefix='Diet').astype(int)
df.drop('DietType', axis=1, inplace=True)
df = pd.concat([df,diet_dummy], axis=1)
x_train, x_test, y_train, y_test = train_test_split(df.drop('Outcome',axis=1), df.Outcome, random_state=20, test_size=0.2)

from sklearn.tree import DecisionTreeClassifier
dtc=DecisionTreeClassifier(max_depth=4)
dtc.fit(x_train,y_train)
print(dtc.score(x_train, y_train))   # training data accuracy
print(dtc.score(x_test, y_test))    # test data accuracy

0.8908794788273615
0.8831168831168831


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gbc = GradientBoostingClassifier()

my_params = {'learning_rate':[0.001,0.003,0.005,0.007,0.009,0.01,0.03,0.05,0.07,0.09,0.1,0.13,0.15,0.17,0.19,0.2,0.23,0.25],
           'n_estimators':[30,50,70,100,130,150],
           'max_depth':[2,3,4,5,6,7,8]}

from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(gbc,param_grid=my_params,cv=5,scoring='accuracy')
grid.fit(x_train,y_train)
grid.best_params_

{'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 70}

In [ ]:
gbc = GradientBoostingClassifier(learning_rate=0.3, max_depth=4, n_estimators=70)
gbc.fit(x_train,y_train)
print(gbc.score(x_train, y_train))   # training data accuracy
print(gbc.score(x_test, y_test))    # test data accuracy

1.0
0.8506493506493507


In [ ]:
from sklearn.ensemble import AdaBoostClassifier
abc = AdaBoostClassifier()

my_params = {'learning_rate':[0.001,0.003,0.005,0.007,0.009,0.01,0.03,0.05,0.07,0.09,0.1,0.13,0.15,0.17,0.19,0.2,0.23,0.25],
           'n_estimators':[30,50,70,100,130,150],
           }

from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(abc,param_grid=my_params,cv=5,scoring='accuracy')
grid.fit(x_train,y_train)
grid.best_params_

{'learning_rate': 0.19, 'n_estimators': 100}

In [ ]:
abc = AdaBoostClassifier(learning_rate=0.19, n_estimators= 100)
abc.fit(x_train,y_train)
print(abc.score(x_train, y_train))   # training data accuracy
print(abc.score(x_test, y_test))    # test data accuracy

0.8697068403908795
0.8896103896103896


# Probability

1. Mutually Exclusive event, p(A $\cap$ B) = 0
2. Independently event p(A|B) = p(A)
3. Law of Addition:
p(A $\cup$ B) = p(A) +  p(B) - p(A $\cap$ B)
4. Special law of addition when p(A $\cap$ B) = 0 mutually exclusive
p(A $\cup$ B) = p(A) + p(B)
5. Law of Multiplication:
p(A $\cap$ B) = P(A) * p(B|A)

if A and B are independent then p(B|A) = p(B)

special law of multiplication:

if A and B are independent then p(B|A) = p(B)
- p(A $\cap$ B) = p(A) * p(B)

In [ ]:
(0.95*0.05)/((0.95*0.05)+(0.95*0.1))

0.3333333333333333